# SHAP for the v2 models

Explains a model trained by `ml/training/train.py`. Features, scaling and the network come from the
same code as training (`ml/features.py`, `FusionModel`), and the feature list is read from the run's
`config.json`, so nothing is recomputed.

The two inputs are explained separately, each with the other held fixed:
1. **Behavioural features**: SHAP KernelExplainer over the 20 features, with the tweet text fixed.
2. **Text**: SHAP's text masker (masking words, not swapping token ids), with the features fixed.

Examples are sampled at random from the test split, from both classes and from correct and wrong
predictions, not only the most confident ones (those were dominated by marketing accounts in v1).

In [ ]:
import sys, json, pathlib, warnings
import numpy as np, pandas as pd, torch, shap
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

PROJECT_ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'ml')); sys.path.insert(0, str(PROJECT_ROOT / 'ml' / 'training'))
from features import load_split, apply_scaler, task_labels
from train import FusionModel
from transformers import AutoTokenizer

RUN_DIR = PROJECT_ROOT / 'out' / 'v2' / 'mental_health__length+sentiment+language__s42'   # <- run to explain
N_EXAMPLES = 40          # test tweets to explain
N_BACKGROUND = 100       # background rows for the feature explainer
SEED = 42

cfg = json.loads((RUN_DIR / 'config.json').read_text())
metrics = json.loads((RUN_DIR / 'metrics.json').read_text())
FEATURES = cfg['features']
THRESHOLD = metrics['test_tweet']['threshold']   # tuned on validation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FusionModel(cfg['base_model'], cfg['num_labels'], len(FEATURES), cfg['dropout']).to(device)
model.load_state_dict(torch.load(RUN_DIR / 'model.pt', map_location=device)); model.eval()
tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
print(f"{cfg['run_name']}: {len(FEATURES)} features, threshold {THRESHOLD}")

In [ ]:
# Test split, model inputs exactly as in training
test_df = load_split('test', FEATURES)
test_df['y'] = task_labels(cfg['task'], test_df['label'].to_numpy())
X_test = apply_scaler(test_df, cfg)

pred = pd.read_parquet(RUN_DIR / 'pred_test.parquet')[['tweet_id', 'p1']]
test_df = test_df.merge(pred, on='tweet_id', how='left')
test_df['pred'] = (test_df['p1'] >= THRESHOLD).astype(int)

# Stratified random sample: true/false positives and negatives, one tweet per user at most
test_df['outcome'] = np.select(
    [(test_df.y == 1) & (test_df.pred == 1), (test_df.y == 0) & (test_df.pred == 0), (test_df.y == 0) & (test_df.pred == 1)],
    ['TP', 'TN', 'FP'], 'FN')
sample = (test_df.sample(frac=1, random_state=SEED).drop_duplicates('user_id')
          .groupby('outcome', group_keys=False).apply(lambda g: g.head(N_EXAMPLES // 4)))
print(sample['outcome'].value_counts().to_string())

In [ ]:
def predict_proba(texts, feats, batch_size=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(list(texts[i:i + batch_size]), truncation=True, max_length=cfg['max_len'],
                            padding=True, return_tensors='pt').to(device)
            delta = torch.as_tensor(feats[i:i + batch_size], dtype=torch.float32, device=device)
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
                logits = model(enc['input_ids'], enc['attention_mask'], delta)
            out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

## 1. Behavioural features (text held fixed)

In [ ]:
rng = np.random.default_rng(SEED)
train_bg = load_split('train', FEATURES, with_text=False)
background = shap.kmeans(apply_scaler(train_bg.sample(5000, random_state=SEED), cfg), 20)

feat_shap = []
for _, row in sample.iterrows():
    x = apply_scaler(row.to_frame().T, cfg)
    f = lambda F: predict_proba([row['text']] * len(F), F)
    explainer = shap.KernelExplainer(f, background)
    feat_shap.append(explainer.shap_values(x, nsamples=200, silent=True)[0])
feat_shap = np.vstack(feat_shap)
X_sample = apply_scaler(sample, cfg)

shap.summary_plot(feat_shap, X_sample, feature_names=FEATURES, show=False)
plt.title('Behavioural features: SHAP on P(positive), text held fixed'); plt.tight_layout(); plt.show()
print(pd.Series(np.abs(feat_shap).mean(0), index=FEATURES).sort_values(ascending=False).round(4).to_string())

## 2. Text (features held fixed)

In [ ]:
text_rows = sample.head(12)
masker = shap.maskers.Text(tokenizer)
text_shap = []
for _, row in text_rows.iterrows():
    x = apply_scaler(row.to_frame().T, cfg)
    f = lambda texts: predict_proba(list(texts), np.repeat(x, len(texts), axis=0))
    explainer = shap.Explainer(f, masker, output_names=['positive'])
    text_shap.append(explainer([row['text']], max_evals=300, silent=True))

for (_, row), sv in zip(text_rows.iterrows(), text_shap):
    print(f"[{row['outcome']}] p={row['p1']:.2f}  user={row['user_id']}")
    shap.plots.text(sv[0])

## 3. How much does the model use the features? (gate values)

In [ ]:
# The gate decides, per tweet, how much of each encoded feature reaches the classifier (0 = ignored)
gates = []
with torch.no_grad():
    for i in range(0, len(sample), 64):
        chunk = sample.iloc[i:i + 64]
        enc = tokenizer(chunk['text'].tolist(), truncation=True, max_length=cfg['max_len'], padding=True, return_tensors='pt').to(device)
        delta = torch.as_tensor(apply_scaler(chunk, cfg), device=device)
        text = model.base(**enc).last_hidden_state[:, 0]
        encoded = model.feature_encoder(delta)
        gates.append(model.fusion_gate(torch.cat([text, encoded], 1)).mean(1).float().cpu().numpy())
sample['mean_gate'] = np.concatenate(gates)
print(sample.groupby('outcome')['mean_gate'].describe().round(3).to_string())